In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =========================================
# STEP 1: INSTALL
# =========================================
!pip install xgboost catboost openpyxl

# =========================================
# STEP 2: IMPORT
# =========================================
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import AdaBoostRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# =========================================
# STEP 3: LOAD DATASET
# =========================================
df = pd.read_excel('/content/drive/MyDrive/DrSeba/Dr.Seba_500_Organized_Final_completed.xlsx')

print("Columns:", df.columns)

# =========================================
# STEP 4: CLEAN TEXT (VERY IMPORTANT)
# =========================================
for col in ['district','specialization_group','hospital_type']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()

# =========================================
# STEP 5: CREATE AUTO MAPPING (NAME → NUMBER)
# =========================================
mappings = {}

for col in ['district','specialization_group','hospital_type']:
    if col in df.columns:
        unique_vals = df[col].unique()
        mapping = {name: idx for idx, name in enumerate(unique_vals)}
        mappings[col] = mapping

        # apply mapping to dataset
        df[col] = df[col].map(mapping)

print("\nMappings:")
for k,v in mappings.items():
    print(k, ":", v)

# =========================================
# STEP 6: ADD FORECASTING FEATURES
# =========================================
np.random.seed(42)

df['date'] = pd.date_range(start='2024-01-01', periods=len(df))
df['total_bookings'] = np.random.randint(20, 200, len(df))
df['commission_rate'] = np.random.uniform(0.05, 0.15, len(df))

df['avg_fee'] = df['Consultation_fees']

# ADD THIS (service_charge from dataset)
df['service_charge'] = df['service_charge']

# UPDATED REVENUE FORMULA
df['revenue'] = (
    df['total_bookings'] * df['avg_fee'] * df['commission_rate']
) + (
    df['total_bookings'] * df['service_charge']
)
# =========================================
# STEP 7: FEATURE ENGINEERING
# =========================================
df['lag_1'] = df['revenue'].shift(1)
df['lag_7'] = df['revenue'].shift(7)
df['rolling_mean_7'] = df['revenue'].rolling(7).mean()

df = df.dropna()

# =========================================
# STEP 8: FEATURES
# =========================================
features = [
    'total_bookings','avg_fee','commission_rate',
    'service_charge',
    'experience_years','rating_avg',
    'district','specialization_group','hospital_type',
    'lag_1','lag_7','rolling_mean_7'
]

features = [f for f in features if f in df.columns]

X = df[features]
y = df['revenue']

# =========================================
# STEP 9: TRAIN SPLIT
# =========================================
split = int(len(df)*0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# =========================================
# STEP 10: TRAIN MODELS
# =========================================
models = {
    "AdaBoost": AdaBoostRegressor(n_estimators=200),
    "XGBoost": XGBRegressor(n_estimators=300, max_depth=6),
    "CatBoost": CatBoostRegressor(iterations=300, verbose=0)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    r2 = r2_score(y_test, preds)
    results[name] = (model, r2)

    print(f"{name} R2:", r2)

best_model_name = max(results, key=lambda x: results[x][1])
best_model = results[best_model_name][0]

print("\n BEST MODEL:", best_model_name)

# =========================================
# STEP 11: INPUT ENCODER (NAME → NUMBER)
# =========================================
def encode_input(user_input):

    encoded = user_input.copy()

    for col in mappings:
        if col in encoded:
            mapping = mappings[col]

            if encoded[col] not in mapping:
                raise ValueError(f"{encoded[col]} not found in {col}")

            encoded[col] = mapping[encoded[col]]

    return encoded

# =========================================
# STEP 12: PREDICT FUNCTION
# =========================================
def predict_revenue(user_input):

    encoded = encode_input(user_input)

    df_input = pd.DataFrame([encoded])
    df_input = df_input[features]

    pred = best_model.predict(df_input)

    return pred[0]

# =========================================
# STEP 13: USER INPUT (NAME BASED)
# =========================================
user_data = {
    'total_bookings': 150,
    'avg_fee': 500,
    'commission_rate': 0.1,
    'service_charge': 50,
    'experience_years': 8,
    'rating_avg': 4.5,

    # NAME INPUT
    'district': 'Dhaka',
    'specialization_group': 'Cardiologist',
    'hospital_type': 'Private Hospital', # Corrected from 'Private' to 'Private Hospital'

    'lag_1': 8000,
    'lag_7': 7500,
    'rolling_mean_7': 7800
}

print("\n💰 FINAL PREDICTION:", predict_revenue(user_data))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.8 MB/s eta 0:00:00
Columns: Index(['thana_rank', 'doctor_id', 'doctor_name', 'district', 'thana',
       'specialization_group', 'experience_years', 'rating_avg',
       'rating_count', 'quality_score', 'hospital_id', 'hospital_type',
       'hospital_name', 'Consultation_fees', 'service_charge',
       'online_consultation', 'emergency_service', 'available_days',
       'available_time', 'full_address', 'date', 'avg_fee', 'commission_rate',
       'revenue', 'email'],
      dtype='object')

Mappings:
district : {'Barisal': 0, 'Chattogram': 1, 'Dhaka': 2, 'Khulna': 3, 'Mymensingh': 4, 'Rajshahi': 5, 'Rangpur': 6, 'Sylhet': 7}
specialization_group : {'Ent Specialist': 0, 'Neurologist': 1, 'Ophthalmologist': 2, 'Surgeon': 3, 'Urologist': 4, 'Pediatrician': 5, 'Dermatologist': 6, 'Medicine Specialist': 7, 'Orthopedist': 8, 'Oncologist': 9, 'Gastroenterologist': 10, 'Cardiologist': 11, 'Gynecologist': 12, 'Dentist': 13, 'Pulmonolog

In [3]:
df.head(400)

,thana_rank,doctor_id,doctor_name,district,thana,specialization_group,experience_years,rating_avg,rating_count,quality_score,...,full_address,date,avg_fee,commission_rate,revenue,email,total_bookings,lag_1,lag_7,rolling_mean_7
7,8,DOC0417,Prof. Dr. F M Siddiqui,0,Bakerganj,5,15,4.21,335,46.80,...,"549 Hospital Lane, Bakerganj, Barisal",2024-01-08,800,0.115572,21039.852921,doctor_416@healthconnect.example,122,13223.190347,12492.929514,17734.679982
8,9,DOC0320,Prof. Dr. Md. Haider Ali Khan,0,Bakerganj,0,23,4.11,174,15.70,...,"491/A Bakerganj Road, Barisal",2024-01-09,2000,0.088540,53168.184711,doctor_319@healthconnect.example,141,21039.852921,21181.095860,22304.264103
9,10,DOC0151,Dr. A.Z.M. Mahfuzur Rahman,0,Bakerganj,6,10,4.09,329,55.22,...,"307/A Bakerganj Road, Barisal",2024-01-10,500,0.118161,10253.584244,doctor_150@healthconnect.example,94,53168.184711,9601.699070,22397.390557
10,11,DOC0220,Dr. Hafiz Ahmed Nazmul Hakim,0,Bakerganj,4,24,4.06,247,54.60,...,"508/A Bakerganj Road, Barisal",2024-01-11,800,0.084062,15755.746315,doctor_219@healthconnect.example,107,10253.584244,12765.361766,22824.588349
11,12,DOC0317,Dr. Hasib Rahman,0,Bakerganj,7,15,4.00,323,31.14,...,"637 Hospital Lane, Bakerganj, Barisal",2024-01-12,600,0.076069,14367.267350,doctor_316@healthconnect.example,136,15755.746315,11634.561280,23214.974931
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
402,4,DOC0375,Dr. Fatema,6,Mithpur,5,19,4.69,77,45.02,...,"479 Medical Plaza, Mithpur, Rangpur - 1358",2025-02-06,1400,0.103648,6842.577341,doctor_374@healthconnect.example,24,24428.440309,9296.573511,15133.436360
403,5,DOC0201,Dr. Prof. Dr. Nasrin,6,Mithpur,8,14,4.60,174,23.60,...,"8 Medical Plaza, Mithpur, Rangpur - 1295",2025-02-07,800,0.087882,18337.298480,doctor_200@healthconnect.example,122,6842.577341,11994.834295,16039.502672
404,6,DOC0467,Dr. Ahmed,6,Mithpur,8,3,4.38,52,29.67,...,"932/A Mithpur Road, Rangpur",2025-02-08,1200,0.095700,5871.000658,doctor_466@healthconnect.example,25,18337.298480,16363.628094,14540.555895
405,7,DOC0277,Dr. Md. Khan,6,Mithpur,4,17,4.33,60,43.56,...,"246 Medical Plaza, Mithpur, Rangpur - 1335",2025-02-09,1000,0.110396,26930.660763,doctor_276@healthconnect.example,128,5871.000658,11955.946044,16679.800855


In [4]:
# =========================================
# MULTI OUTPUT (DAY / WEEK / MONTH)
# =========================================

def multi_forecast(user_input, days=30):

    current_input = encode_input(user_input.copy())
    preds = []

    for i in range(days):

        df_input = pd.DataFrame([current_input])
        df_input = df_input[features]

        pred = best_model.predict(df_input)[0]
        preds.append(pred)

        # Update lag features
        current_input['lag_7'] = current_input['lag_1']
        current_input['lag_1'] = pred

        current_input['rolling_mean_7'] = (
            current_input['rolling_mean_7'] * 6 + pred
        ) / 7

    return preds


# =========================================
#  NEW USER INPUT (UPDATED WITH SERVICE CHARGE)
# =========================================
user_input_test = {
    'total_bookings': 180,
    'avg_fee': 700,
    'commission_rate': 0.12,
    'service_charge': 60,
    'experience_years': 12,
    'rating_avg': 4.7,

    'district': 'Chattogram',
    'specialization_group': 'Neurologist',
    'hospital_type': 'Private Hospital',

    'lag_1': 9500,
    'lag_7': 8800,
    'rolling_mean_7': 9000
}


# =========================================
# RUN FORECAST
# =========================================
forecast = multi_forecast(user_input_test, days=30)


# =========================================
#  DAILY OUTPUT
# =========================================
print("\n DAILY FORECAST (Next 7 Days):")
for i in range(7):
    print(f"Day {i+1}: {round(forecast[i],2)}")


# =========================================
#  WEEKLY OUTPUT
# =========================================
weekly = sum(forecast[:7])
print("\n WEEKLY FORECAST:", round(weekly,2))


# =========================================
#  MONTHLY OUTPUT
# =========================================
monthly = sum(forecast)
print("\n MONTHLY FORECAST:", round(monthly,2))


 DAILY FORECAST (Next 7 Days):
Day 1: 23899.81
Day 2: 24214.95
Day 3: 23997.51
Day 4: 23997.51
Day 5: 24004.76
Day 6: 24004.76
Day 7: 24147.33

 WEEKLY FORECAST: 168266.63

 MONTHLY FORECAST: 730116.45


In [ ]:
# =========================================
# MULTI OUTPUT (DAY / WEEK / MONTH)
# =========================================

def multi_forecast(user_input, days=30):

    current_input = encode_input(user_input.copy())
    preds = []

    for i in range(days):

        df_input = pd.DataFrame([current_input])
        df_input = df_input[features]

        pred = best_model.predict(df_input)[0]
        preds.append(pred)

        # ❁ Update lag features
        current_input['lag_7'] = current_input['lag_1']
        current_input['lag_1'] = pred

        current_input['rolling_mean_7'] = (
            current_input['rolling_mean_7'] * 6 + pred
        ) / 7

    return preds


# =========================================
#  NEW USER INPUT (DIFFERENT EXAMPLE)
# =========================================
user_input_test = {
    'total_bookings': 100,
    'avg_fee': 400,
    'commission_rate': 0.10,
    'service_charge': 60,
    'experience_years': 12,
    'rating_avg': 4.0,

    'district': 'Chattogram', # Corrected from 'Chittagong' to 'Chattogram'
    'specialization_group': 'Neurologist',
    'hospital_type': 'Private Hospital',

    'lag_1': 9500,
    'lag_7': 8800,
    'rolling_mean_7': 9000
}


# =========================================
# RUN FORECAST
# =========================================
forecast = multi_forecast(user_input_test, days=30)


# =========================================
#  DAILY OUTPUT
# =========================================
print("\n DAILY FORECAST (Next 7 Days):")
for i in range(7):
    print(f"Day {i+1}: {round(forecast[i],2)}")


# =========================================
#  WEEKLY OUTPUT
# =========================================
weekly = sum(forecast[:7])
print("\n WEEKLY FORECAST:", round(weekly,2))


# =========================================
# MONTHLY OUTPUT
# =========================================
monthly = sum(forecast)
print("\n MONTHLY FORECAST:", round(monthly,2))


 DAILY FORECAST (Next 7 Days):
Day 1: 11857.01
Day 2: 11847.94
Day 3: 11873.36
Day 4: 11873.36
Day 5: 11873.36
Day 6: 11873.36
Day 7: 11873.36

 WEEKLY FORECAST: 83071.74

 MONTHLY FORECAST: 356158.93


In [ ]:
import pickle

model_package = {
    "model": best_model,   # BEST MODEL
    "mappings": mappings, # Use mappings dictionary for encoding
    "features": features
}

with open("commission_revenue_forecasting_model.pkl", "wb") as f:
    pickle.dump(model_package, f)

print("Full model package saved!")

Full model package saved!


In [ ]:
from google.colab import files
files.download("commission_revenue_forecasting_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_path = "/content/drive/MyDrive/DrSeba/commission_revenue_forecasting_model.pkl"

with open(drive_path, "wb") as f:
    pickle.dump(model_package, f)

print("Saved to Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Google Drive!
